# EDA: ML Training Data

Exploratory data analysis of `ml_training_data_syd_mel.csv` with focus on ML-relevant insights.

**Dataset:** Sydney ↔ Melbourne flight delays with weather features  
**Targets:** `delay_rate` (regression), `is_high_delay` (classification)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

%matplotlib inline

## 1. Data Overview & Quality Check

In [ ]:
# Load data
df = pd.read_csv('../data/processed/ml_training_data_syd_mel.csv')
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# Preview data
df.head()

In [ ]:
# Missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values")
else:
    print(missing[missing > 0])

In [ ]:
# Duplicates
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

In [ ]:
# Numerical summary
df.describe().T

## 2. Target Variable Analysis

### 2.1 Regression Target: `delay_rate`

In [ ]:
# Distribution statistics
print("delay_rate statistics:")
print(f"  Mean:     {df['delay_rate'].mean():.4f} ({df['delay_rate'].mean()*100:.2f}%)")
print(f"  Median:   {df['delay_rate'].median():.4f}")
print(f"  Std Dev:  {df['delay_rate'].std():.4f}")
print(f"  Min:      {df['delay_rate'].min():.4f}")
print(f"  Max:      {df['delay_rate'].max():.4f}")
print(f"  Skewness: {df['delay_rate'].skew():.4f}")
print(f"  Kurtosis: {df['delay_rate'].kurtosis():.4f}")

In [ ]:
# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with KDE
axes[0].hist(df['delay_rate'], bins=30, density=True, alpha=0.7, edgecolor='black')
df['delay_rate'].plot(kind='kde', ax=axes[0], color='red', linewidth=2)
axes[0].axvline(df['delay_rate'].mean(), color='blue', linestyle='--', label=f'Mean: {df["delay_rate"].mean():.3f}')
axes[0].axvline(df['delay_rate'].median(), color='green', linestyle='--', label=f'Median: {df["delay_rate"].median():.3f}')
axes[0].set_xlabel('Delay Rate')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of delay_rate')
axes[0].legend()

# Q-Q plot
stats.probplot(df['delay_rate'], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normal)')

plt.tight_layout()
plt.show()

In [ ]:
# Normality test
stat, p_value = stats.shapiro(df['delay_rate'].sample(min(5000, len(df)), random_state=42))
print(f"Shapiro-Wilk test: W={stat:.4f}, p={p_value:.4e}")
print(f"Normal distribution: {'Yes' if p_value > 0.05 else 'No (consider transformation)'}")

### 2.2 Classification Target: `is_high_delay`

In [ ]:
# Class distribution
class_counts = df['is_high_delay'].value_counts()
class_pct = df['is_high_delay'].value_counts(normalize=True) * 100

print("Class distribution:")
print(f"  0 (On-time):    {class_counts[0]:,} ({class_pct[0]:.1f}%)")
print(f"  1 (High-delay): {class_counts[1]:,} ({class_pct[1]:.1f}%)")
print(f"  Imbalance ratio: {class_counts[0]/class_counts[1]:.2f}:1")

In [ ]:
# Class balance visualization
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['On-time (0)', 'High-delay (1)'], class_counts.values, 
              color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.bar_label(bars, labels=[f'{v:,}\n({p:.1f}%)' for v, p in zip(class_counts.values, class_pct.values)])
ax.set_ylabel('Count')
ax.set_title('Classification Target Balance')
ax.axhline(len(df)/2, color='gray', linestyle='--', alpha=0.5, label='Perfect balance')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
# Identify weather feature columns
weather_cols_dep = [c for c in df.columns if c.endswith('_dep')]
weather_cols_arr = [c for c in df.columns if c.endswith('_arr')]
weather_cols = weather_cols_dep + weather_cols_arr

print(f"Weather features: {len(weather_cols)} total")
print(f"  - Departure: {len(weather_cols_dep)}")
print(f"  - Arrival: {len(weather_cols_arr)}")

In [ ]:
# Skewness of weather features
skewness = df[weather_cols].skew().sort_values(key=abs, ascending=False)
print("Most skewed features (|skew| > 1 may need transformation):")
print(skewness[abs(skewness) > 1].to_string())

In [ ]:
# Distribution of key weather features (departure)
key_features_dep = ['avg_max_temp_dep', 'avg_rainfall_per_day_dep', 'avg_wind_speed_dep', 
                    'avg_max_humidity_dep', 'extreme_weather_days_dep']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_features_dep):
    axes[i].hist(df[col], bins=25, edgecolor='black', alpha=0.7)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='Mean')
    axes[i].set_xlabel(col.replace('_dep', ''))
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{col}\nskew={df[col].skew():.2f}')

axes[-1].axis('off')
plt.suptitle('Key Weather Features (Departure Airport)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection via box plots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_features_dep):
    axes[i].boxplot(df[col], vert=True)
    axes[i].set_ylabel(col.replace('_dep', ''))
    
    # Count outliers (IQR method)
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    axes[i].set_title(f'{col}\n{outliers} outliers ({outliers/len(df)*100:.1f}%)')

axes[-1].axis('off')
plt.suptitle('Outlier Detection (Departure Weather)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature-Target Relationships

In [ ]:
# Correlation with delay_rate
corr_with_target = df[weather_cols + ['delay_rate']].corr()['delay_rate'].drop('delay_rate')
corr_sorted = corr_with_target.sort_values(key=abs, ascending=False)

print("Top 15 features correlated with delay_rate:")
print(corr_sorted.head(15).to_string())

In [ ]:
# Correlation bar plot
fig, ax = plt.subplots(figsize=(10, 8))
top_corr = corr_sorted.head(20)
colors = ['#e74c3c' if x > 0 else '#3498db' for x in top_corr.values]
ax.barh(range(len(top_corr)), top_corr.values, color=colors, edgecolor='black')
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index)
ax.set_xlabel('Correlation with delay_rate')
ax.set_title('Top 20 Feature Correlations with Target')
ax.axvline(0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for top correlated features
top_features = corr_sorted.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df['delay_rate'], alpha=0.3, s=10)
    # Add trend line
    z = np.polyfit(df[col], df['delay_rate'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r-', linewidth=2)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('delay_rate')
    r = corr_with_target[col]
    axes[i].set_title(f'r = {r:.3f}')

plt.suptitle('Top Correlated Features vs delay_rate', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: features grouped by is_high_delay
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    df.boxplot(column=col, by='is_high_delay', ax=axes[i])
    axes[i].set_xlabel('is_high_delay')
    axes[i].set_ylabel(col)
    axes[i].set_title(col)

plt.suptitle('Feature Distributions by High Delay Class', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Correlations & Multicollinearity

In [ ]:
# Correlation matrix heatmap (departure features only for clarity)
corr_matrix_dep = df[weather_cols_dep].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix_dep, dtype=bool))
sns.heatmap(corr_matrix_dep, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix (Departure Weather)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated feature pairs (|r| > 0.8)
def get_high_correlations(corr_matrix, threshold=0.8):
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                pairs.append((cols[i], cols[j], corr_matrix.iloc[i, j]))
    return sorted(pairs, key=lambda x: abs(x[2]), reverse=True)

high_corr_pairs = get_high_correlations(df[weather_cols].corr(), threshold=0.8)

print(f"Highly correlated feature pairs (|r| > 0.8): {len(high_corr_pairs)} pairs")
print("\nTop pairs to consider for feature selection:")
for f1, f2, r in high_corr_pairs[:15]:
    print(f"  {r:+.3f}: {f1} ↔ {f2}")

In [ ]:
# Correlation between departure and arrival features
dep_arr_corr = []
for col in weather_cols_dep:
    base_name = col.replace('_dep', '')
    arr_col = base_name + '_arr'
    if arr_col in df.columns:
        r = df[col].corr(df[arr_col])
        dep_arr_corr.append((base_name, r))

dep_arr_corr = sorted(dep_arr_corr, key=lambda x: x[1], reverse=True)
print("Correlation between departure and arrival features:")
print("(High correlation suggests redundancy)\n")
for name, r in dep_arr_corr:
    print(f"  {r:.3f}: {name}")

## 6. Temporal Patterns

In [ ]:
# Monthly seasonality
monthly_delay = df.groupby('month')['delay_rate'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_delay.index, monthly_delay['mean'], yerr=monthly_delay['std'], 
       capsize=3, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Average Delay Rate')
ax.set_title('Monthly Seasonality in Delay Rates')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.axhline(df['delay_rate'].mean(), color='red', linestyle='--', label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Year-over-year trends
yearly_delay = df.groupby('year')['delay_rate'].agg(['mean', 'std', 'count'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly_delay.index, yearly_delay['mean'], marker='o', linewidth=2, markersize=8)
ax.fill_between(yearly_delay.index, 
                yearly_delay['mean'] - yearly_delay['std'],
                yearly_delay['mean'] + yearly_delay['std'],
                alpha=0.3)
ax.set_xlabel('Year')
ax.set_ylabel('Average Delay Rate')
ax.set_title('Year-over-Year Delay Rate Trends')
ax.axhline(df['delay_rate'].mean(), color='red', linestyle='--', alpha=0.5, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Time series of delay rate
df['year_month_dt'] = pd.to_datetime(df['year_month'])
ts_delay = df.groupby('year_month_dt')['delay_rate'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_delay.index, ts_delay.values, linewidth=1, alpha=0.8)
ax.axhline(0.25, color='red', linestyle='--', label='High delay threshold (25%)')
ax.set_xlabel('Date')
ax.set_ylabel('Average Delay Rate')
ax.set_title('Delay Rate Time Series (All Airlines)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Categorical Variable Analysis

In [ ]:
# Delay rate by airline
airline_stats = df.groupby('airline').agg({
    'delay_rate': ['mean', 'std', 'count'],
    'is_high_delay': 'mean'
}).round(4)
airline_stats.columns = ['mean_delay', 'std_delay', 'n_records', 'pct_high_delay']
airline_stats = airline_stats.sort_values('mean_delay', ascending=False)
print(airline_stats)

In [ ]:
# Airline delay rate visualization
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(airline_stats)))
bars = ax.barh(airline_stats.index, airline_stats['mean_delay'], 
               xerr=airline_stats['std_delay'], capsize=3,
               color=colors, edgecolor='black')
ax.set_xlabel('Average Delay Rate')
ax.set_title('Delay Rate by Airline')
ax.axvline(0.25, color='red', linestyle='--', alpha=0.7, label='High delay threshold')
ax.axvline(df['delay_rate'].mean(), color='blue', linestyle='--', alpha=0.7, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Delay rate by route direction
route_stats = df.groupby(['departing_port', 'arriving_port']).agg({
    'delay_rate': ['mean', 'std', 'count'],
    'is_high_delay': 'mean'
}).round(4)
route_stats.columns = ['mean_delay', 'std_delay', 'n_records', 'pct_high_delay']
print(route_stats)

In [ ]:
# Statistical test: is there a significant difference between routes?
syd_mel = df[df['departing_port'] == 'Sydney']['delay_rate']
mel_syd = df[df['departing_port'] == 'Melbourne']['delay_rate']

stat, p_value = stats.mannwhitneyu(syd_mel, mel_syd, alternative='two-sided')
print(f"Mann-Whitney U test (Sydney→Melbourne vs Melbourne→Sydney):")
print(f"  U statistic: {stat:.2f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
# Record counts by airline (for sample weighting considerations)
fig, ax = plt.subplots(figsize=(10, 5))
airline_counts = df['airline'].value_counts()
ax.bar(airline_counts.index, airline_counts.values, color='steelblue', edgecolor='black')
ax.set_xlabel('Airline')
ax.set_ylabel('Number of Records')
ax.set_title('Sample Size by Airline')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(airline_counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

### ML-Relevant Insights

#### Target Variables
- **Regression (`delay_rate`):** Right-skewed distribution (skew > 0), may benefit from log or Box-Cox transformation for linear models
- **Classification (`is_high_delay`):** Moderately imbalanced (~57:43), may not require aggressive resampling but consider stratified splits

#### Feature Quality
- **No missing values** - data is clean and ready for modeling
- **Several highly skewed features** (rainfall, extreme weather days) - consider transformation for linear models
- **Some outliers present** but within plausible meteorological ranges

#### Feature Relationships
- Weather features show **weak to moderate correlations** with delay rate (typically |r| < 0.3)
- Top predictive features identified for feature selection
- **High multicollinearity** between some feature pairs - consider:
  - Regularization (Ridge/Lasso)
  - Feature selection or PCA
  - Removing redundant features

#### Departure vs Arrival Features
- High correlation between departure and arrival weather (same route, similar climate)
- However, this is likely route specific since Sydney is relatively close to Melbourne.

#### Temporal Patterns
- Clear monthly seasonality. Will add cyclical encoding to add month as a feature.
- Year-over-year variation exists. Will use year-based train/test split

#### Categorical Variables
- **Airline** shows significant variation in delay rates - important predictor
- Route direction shows some difference - may be useful feature
- Unequal sample sizes across airlines - consider stratified sampling

## 9. Next step

Try adding holiday features that quantifies the number of public holidays and the occurence of a major holiday in each month. Extremely busy periods might cause delays.

See [3b_engineer_holiday_features.ipynb](../notebooks/3b_engineer_holiday_features.ipynb)